In [13]:
!pip install datasets

In [14]:
!pip install transformers

In [15]:
# =========================
# 1. IMPORTS
# =========================
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix

In [16]:


# =========================
# 2. DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [17]:
# =========================
# 3. LOAD DATASET
# =========================
dataset = load_dataset("hate_speech_offensive")
dataset = dataset["train"]

# Convert to binary labels:
# class: 0 = hate, 1 = offensive, 2 = neither
# we map:
# 2 -> normal (0)
# others -> hate/offensive (1)

def convert(example):
    example["label"] = 0 if example["class"] == 2 else 1
    return example

dataset = dataset.map(convert)

# =========================
# 4. TRAIN / TEST SPLIT
# =========================
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_data = dataset["train"]
test_data = dataset["test"]

In [18]:
# =========================
# 5. TOKENIZER
# =========================
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["tweet"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_data = train_data.map(tokenize)
test_data = test_data.map(tokenize)

# =========================
# 6. CUSTOM COLLATE FUNCTION (IMPORTANT FIX)
# =========================
def collate_fn(batch):
    return {
        "input_ids": torch.tensor([x["input_ids"] for x in batch]),
        "attention_mask": torch.tensor([x["attention_mask"] for x in batch]),
        "label": torch.tensor([x["label"] for x in batch])
    }

# =========================
# 7. DATALOADERS
# =========================
train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=16, collate_fn=collate_fn)

# =========================
# 8. MODEL
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
).to(device)

# =========================
# 9. OPTIMIZER + LOSS
# =========================
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

# =========================
# 10. TRAINING LOOP
# =========================
epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 10 == 0:
            print(f"Epoch {epoch} | Batch {i} | Loss: {loss.item():.4f}")

    print(f"Epoch {epoch} Average Loss: {total_loss/len(train_loader):.4f}")

save_path = "./sentiment_classifier"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
# =========================
# 11. EVALUATION
# =========================
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# =========================
# 12. CLASSIFICATION REPORT
# =========================
print("\nCLASSIFICATION REPORT:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["normal", "hate/offensive"]
))

# =========================
# 13. CONFUSION MATRIX
# =========================
print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_true, y_pred))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 0 | Batch 0 | Loss: 0.6178
Epoch 0 | Batch 10 | Loss: 0.6885
Epoch 0 | Batch 20 | Loss: 0.6133
Epoch 0 | Batch 30 | Loss: 0.4455
Epoch 0 | Batch 40 | Loss: 0.2950
Epoch 0 | Batch 50 | Loss: 0.1813
Epoch 0 | Batch 60 | Loss: 0.1991
Epoch 0 | Batch 70 | Loss: 0.7236
Epoch 0 | Batch 80 | Loss: 0.2807
Epoch 0 | Batch 90 | Loss: 0.0558
Epoch 0 | Batch 100 | Loss: 0.0902
Epoch 0 | Batch 110 | Loss: 0.1607
Epoch 0 | Batch 120 | Loss: 0.1091
Epoch 0 | Batch 130 | Loss: 0.1871
Epoch 0 | Batch 140 | Loss: 0.4709
Epoch 0 | Batch 150 | Loss: 0.0294
Epoch 0 | Batch 160 | Loss: 0.5747
Epoch 0 | Batch 170 | Loss: 0.4191
Epoch 0 | Batch 180 | Loss: 0.1463
Epoch 0 | Batch 190 | Loss: 0.0633
Epoch 0 | Batch 200 | Loss: 0.1217
Epoch 0 | Batch 210 | Loss: 0.0876
Epoch 0 | Batch 220 | Loss: 0.0968
Epoch 0 | Batch 230 | Loss: 0.1479
Epoch 0 | Batch 240 | Loss: 0.0312
Epoch 0 | Batch 250 | Loss: 0.0177
Epoch 0 | Batch 260 | Loss: 0.1757
Epoch 0 | Batch 270 | Loss: 0.0338
Epoch 0 | Batch 280 | Loss: 0.0

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


CLASSIFICATION REPORT:
                precision    recall  f1-score   support

        normal       0.94      0.81      0.87       858
hate/offensive       0.96      0.99      0.97      4099

      accuracy                           0.96      4957
     macro avg       0.95      0.90      0.92      4957
  weighted avg       0.96      0.96      0.96      4957


CONFUSION MATRIX:
[[ 694  164]
 [  45 4054]]
